# EcoMaps · Entrenamiento del clasificador de residuos

Aprendizaje por transferencia con **MobileNetV2** sobre TrashNet, exportado a **TensorFlow.js** para correrlo en el navegador (RF-01, RF-08).

**Clases (solo lo que EcoMaps recibe en el mapa):** `metal`, `papel_carton`, `plastico`, `vidrio`.
**Datos:** 300 imágenes de entrenamiento y 60 de prueba por clase. Las de prueba nunca se usan para entrenar.

**Cómo usarlo en Google Colab**
1. Menú *Entorno de ejecución → Cambiar tipo de entorno → GPU* (más rápido).
2. *Entorno de ejecución → Ejecutar todas*. Si Colab pide reiniciar tras instalar las librerías, reinicia y vuelve a ejecutar todo.
3. Al final se descarga `modelo_v1.zip`: descomprímelo en `app/static/modelo/v1/` del proyecto.

In [ ]:
# Keras 2 (tf_keras): el exportador a TensorFlow.js lo necesita. Debe ir ANTES de importar TensorFlow.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

In [ ]:
!pip install -q tf_keras tensorflowjs datasets pillow scikit-learn

In [ ]:
import json, random, shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tf_keras as keras
from tf_keras import layers

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); tf.random.set_seed(SEMILLA)

# Clase de TrashNet -> clase de EcoMaps ('trash' se descarta a propósito)
MAPA = {"plastic": "plastico", "glass": "vidrio", "metal": "metal",
        "paper": "papel_carton", "cardboard": "papel_carton"}
N_ENTRENAMIENTO, N_PRUEBA = 300, 60
IMG, LOTE = 224, 16
LADO_MAX = 512
DATA = Path("dataset")
UMBRAL = 0.70  # RF-04: por debajo, la página le pregunta al usuario

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 1. Dataset
Se descarga TrashNet de Hugging Face (Colab tiene internet) y se arma el reparto entrenamiento/prueba.
Si prefieres usar tu propia carpeta o un enlace distinto, cambia `ORIGEN_LOCAL` (carpeta con subcarpetas `cardboard`, `glass`, `metal`, `paper`, `plastic`, `trash`) o el nombre del dataset en la celda.

In [ ]:
ORIGEN_LOCAL = None   # ej: "/content/drive/MyDrive/trashnet/dataset-resized"  (deja None para usar Hugging Face)
DATASET_HF = "garythung/trashnet"

from PIL import Image

def reunir():
    """Devuelve {clase_ecomaps: [imagen PIL o ruta, ...]}."""
    grupos = {c: [] for c in sorted(set(MAPA.values()))}
    if ORIGEN_LOCAL:
        for sub in Path(ORIGEN_LOCAL).iterdir():
            clase = MAPA.get(sub.name.lower())
            if clase and sub.is_dir():
                grupos[clase] += [p for p in sub.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}]
    else:
        from datasets import load_dataset
        ds = load_dataset(DATASET_HF)          # igual que en Hugging Face: devuelve un DatasetDict
        print(ds)
        ds = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]
        nombres = ds.features["label"].names
        print("Etiquetas originales:", nombres)
        etiquetas = ds["label"]  # solo las etiquetas: no decodifica todas las imágenes
        for i, e in enumerate(etiquetas):
            clase = MAPA.get(nombres[e].lower())
            if clase:
                grupos[clase].append(i)
        grupos = {c: [(ds, i) for i in idx] for c, idx in grupos.items()}
    return grupos

def abrir(item):
    if isinstance(item, tuple):            # (dataset HF, índice)
        return item[0][item[1]]["image"]
    return Image.open(item)

if DATA.exists():
    shutil.rmtree(DATA)
grupos = reunir()
rnd = random.Random(SEMILLA)
for clase, items in sorted(grupos.items()):
    rnd.shuffle(items)
    n_prueba = min(N_PRUEBA, len(items) // 5)
    n_entr = min(N_ENTRENAMIENTO, len(items) - n_prueba)
    reparto = {"prueba": items[:n_prueba], "entrenamiento": items[n_prueba:n_prueba + n_entr]}
    for parte, lista in reparto.items():
        carpeta = DATA / parte / clase
        carpeta.mkdir(parents=True, exist_ok=True)
        for k, it in enumerate(lista):
            im = abrir(it).convert("RGB")
            im.thumbnail((LADO_MAX, LADO_MAX))
            im.save(carpeta / f"{clase}_{k:03d}.jpg", quality=85)
    print(f"{clase}: {n_entr} entrenamiento + {n_prueba} prueba (disponibles: {len(items)})")

## 2. Datos para Keras
El aumento de datos (giros, zoom, contraste) solo se aplica al entrenar. El 15 % del entrenamiento se separa para validar.

In [ ]:
comunes = dict(image_size=(IMG, IMG), batch_size=LOTE, label_mode="categorical", seed=SEMILLA)
train_raw = keras.utils.image_dataset_from_directory(DATA / "entrenamiento", validation_split=0.15, subset="training", **comunes)
val_raw   = keras.utils.image_dataset_from_directory(DATA / "entrenamiento", validation_split=0.15, subset="validation", **comunes)
test_raw  = keras.utils.image_dataset_from_directory(DATA / "prueba", shuffle=False, **comunes)
CLASES = train_raw.class_names           # orden alfabético; se guarda en metadata.json
print("Clases:", CLASES)

prep = keras.applications.mobilenet_v2.preprocess_input   # escala a [-1, 1]; la página hace lo mismo
aumento = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.15),
])
AUTO = tf.data.AUTOTUNE
train = train_raw.map(lambda x, y: (prep(aumento(x, training=True)), y), num_parallel_calls=AUTO).prefetch(AUTO)
val   = val_raw.map(lambda x, y: (prep(x), y), num_parallel_calls=AUTO).prefetch(AUTO)
test  = test_raw.map(lambda x, y: (prep(x), y), num_parallel_calls=AUTO).prefetch(AUTO)

## 3. Modelo (transferencia desde MobileNetV2)
Primero se entrena solo la capa final; luego se afinan las últimas capas de MobileNetV2 con una tasa de aprendizaje baja.

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base.trainable = False

entrada = keras.Input((IMG, IMG, 3))
x = base(entrada, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
salida = layers.Dense(len(CLASES), activation="softmax")(x)
modelo = keras.Model(entrada, salida)

alto = keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True)

modelo.compile(optimizer=keras.optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
h1 = modelo.fit(train, validation_data=val, epochs=15, callbacks=[alto])

# Ajuste fino: se liberan las últimas 30 capas de MobileNetV2
base.trainable = True
for capa in base.layers[:-30]:
    capa.trainable = False
modelo.compile(optimizer=keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
h2 = modelo.fit(train, validation_data=val, epochs=10, callbacks=[alto])

In [ ]:
acc = h1.history["accuracy"] + h2.history["accuracy"]
vacc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
plt.figure(figsize=(6, 3.5))
plt.plot(acc, label="entrenamiento"); plt.plot(vacc, label="validación")
plt.axvline(len(h1.history["accuracy"]) - 1, ls="--", c="gray", label="inicio ajuste fino")
plt.xlabel("época"); plt.ylabel("exactitud"); plt.legend(); plt.tight_layout()
plt.savefig("curvas_entrenamiento.png", dpi=150); plt.show()

## 4. Evaluación con las imágenes de prueba (RF-08)
Estas imágenes no se usaron para entrenar. Meta orientativa: exactitud ≥ 80 %.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_true = np.concatenate([y.numpy().argmax(1) for _, y in test])
probs = modelo.predict(test)
y_pred = probs.argmax(1)
exactitud = float((y_true == y_pred).mean())
print(f"Exactitud global: {exactitud:.1%}  ({int((y_true == y_pred).sum())} de {len(y_true)} imágenes de prueba)\n")
print(classification_report(y_true, y_pred, target_names=CLASES, digits=3))

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(len(CLASES))); ax.set_xticklabels(CLASES, rotation=30, ha="right")
ax.set_yticks(range(len(CLASES))); ax.set_yticklabels(CLASES)
ax.set_xlabel("Clase predicha"); ax.set_ylabel("Clase real"); ax.set_title(f"Matriz de confusión · exactitud {exactitud:.1%}")
for i in range(len(CLASES)):
    for j in range(len(CLASES)):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout(); plt.savefig("matriz_confusion.png", dpi=150); plt.show()

# Pares que más se confunden (para el informe: qué se hizo para mejorarlos)
confusiones = [(CLASES[i], CLASES[j], int(cm[i, j])) for i in range(len(CLASES)) for j in range(len(CLASES)) if i != j and cm[i, j] > 0]
confusiones.sort(key=lambda t: -t[2])
print("Confusiones más frecuentes (real -> predicha):", confusiones[:5])

# Cuántas predicciones quedarían bajo el umbral y la página preguntaría al usuario
dudosas = float((probs.max(1) < UMBRAL).mean())
print(f"Predicciones con confianza < {UMBRAL:.0%} (la página pregunta al usuario): {dudosas:.1%}")

json.dump({"exactitud": exactitud, "clases": CLASES, "matriz_confusion": cm.tolist(),
           "imagenes_prueba": int(len(y_true)), "confusiones_top": confusiones[:5], "pct_bajo_umbral": dudosas},
          open("informe_evaluacion.json", "w"), indent=2, ensure_ascii=False)

## 5. Exportar a TensorFlow.js
Genera `model.json`, los archivos `.bin` y `metadata.json` (con el nombre de cada clase, en el mismo orden que usa el modelo).

In [ ]:
import tensorflowjs as tfjs

SALIDA = Path("modelo_v1")
if SALIDA.exists():
    shutil.rmtree(SALIDA)
tfjs.converters.save_keras_model(modelo, str(SALIDA))
json.dump({"labels": CLASES, "imageSize": IMG, "preprocess": "x/127.5 - 1", "umbral": UMBRAL, "version": "v1"},
          open(SALIDA / "metadata.json", "w"), indent=2, ensure_ascii=False)

print("Archivos exportados:")
for f in sorted(SALIDA.iterdir()):
    print(f"  {f.name}  {f.stat().st_size / 1e6:.2f} MB")

shutil.copy("matriz_confusion.png", SALIDA.parent / "matriz_confusion.png")
shutil.make_archive("modelo_v1", "zip", SALIDA)

In [ ]:
# Descarga el modelo y los resultados (en Colab). Descomprime modelo_v1.zip en app/static/modelo/v1/
try:
    from google.colab import files
    for f in ["modelo_v1.zip", "matriz_confusion.png", "informe_evaluacion.json", "curvas_entrenamiento.png"]:
        files.download(f)
except ImportError:
    print("Fuera de Colab: los archivos quedaron en la carpeta actual.")

## 6. Prueba con una imagen nueva (opcional)
Sube una foto de un residuo y mira qué responde, con la misma regla de la página: si la confianza es menor al 70 %, muestra las 2 opciones más probables.

In [ ]:
from google.colab import files
subida = files.upload()
for nombre in subida:
    im = Image.open(nombre).convert("RGB").resize((IMG, IMG))
    x = prep(np.array(im, dtype="float32"))[None]
    p = modelo.predict(x, verbose=0)[0]
    orden = p.argsort()[::-1]
    if p[orden[0]] >= UMBRAL:
        print(f"{nombre}: {CLASES[orden[0]]} ({p[orden[0]]:.0%})")
    else:
        print(f"{nombre}: no estoy seguro -> {CLASES[orden[0]]} ({p[orden[0]]:.0%}) o {CLASES[orden[1]]} ({p[orden[1]]:.0%})")
    plt.imshow(im); plt.axis("off"); plt.show()